# Notebook: nb_silver_customers
# Purpose: Clean + conform bronze_customers into silver_customers
# Layer: Silver (clean / conform)
# Source: bronze_customers (read EXCLUSIVELY via read_bronze)
# Target: silver_customers (full-refresh overwrite)

In [ ]:
%run utilities/nb_utils_config

In [ ]:
# --- Imports ---
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [ ]:
# --- Configuration ---
TABLE_NAME = silver_table("customers")
BRONZE_SOURCE = "customers"

In [ ]:
# --- Read from bronze layer (the ONLY allowed read pattern) ---
df_raw = read_bronze(BRONZE_SOURCE)

print(f"Bronze rows: {df_raw.count():,}")
print(f"Bronze columns: {len(df_raw.columns)}")

In [ ]:
# --- Rename + cast ---
df_clean = df_raw \
    .withColumnRenamed("CustomerID", "customer_id") \
    .withColumnRenamed("FullName", "full_name") \
    .withColumnRenamed("SignupDate", "signup_date") \
    .withColumnRenamed("LifetimeValue", "lifetime_value") \
    .withColumnRenamed("StatusCode", "status_code") \
    .withColumn("customer_id", F.col("customer_id").cast("int")) \
    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd")) \
    .withColumn("lifetime_value", F.col("lifetime_value").cast("decimal(19,4)"))

In [ ]:
# --- Decode categorical + handle nulls ---
df_clean = df_clean \
    .withColumn("status", F.when(F.col("status_code") == "A", "Active")
                          .when(F.col("status_code") == "I", "Inactive")
                          .otherwise("Unknown")) \
    .fillna({"lifetime_value": 0, "full_name": "Unknown"})

In [ ]:
# --- Deduplicate: keep the latest row per customer_id ---
w = Window.partitionBy("customer_id").orderBy(F.col("_load_timestamp").desc())
df_clean = df_clean \
    .withColumn("_row_num", F.row_number().over(w)) \
    .filter(F.col("_row_num") == 1) \
    .drop("_row_num")

In [ ]:
# --- Filter invalid / test rows ---
df_clean = df_clean \
    .filter(~F.col("full_name").rlike("(?i)^test")) \
    .filter(F.col("lifetime_value") >= 0)

In [ ]:
# --- Drop bronze metadata + add silver metadata ---
df_silver = df_clean \
    .drop("_load_timestamp", "_source_file", "_load_id")
df_silver = add_silver_metadata(df_silver)

In [ ]:
# --- Write to silver Delta table (overwrite, NOT append) ---
df_silver.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(TABLE_NAME)

print(f"Written to {TABLE_NAME}")

In [ ]:
# --- Validation ---
validate_row_count(TABLE_NAME)
print("PASS: Silver transform complete")